## homework 4

In [1]:
import torch
import torch.nn as nn

from torch.nn import Sequential
from torch.nn import GRU, LSTM
from torch.utils.data import DataLoader, TensorDataset
from torch import tensor

import numpy as np
import pandas as pd
import yfinance

import time

from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import shuffle

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


1. download and preprocess the data
- sliding window to collect sequences


TODO may need to go back and check for bad values that are not null. 

In [ ]:
# do this here if needed... download for 2021
# 80/20 split for data

M = 60
N = 1

rstate = np.random.RandomState(1)


def preprocess(data : pd.DataFrame, batch_size_ = 10):
    data_close = data["Close"].to_numpy()
    X_seq = []
    y_seq = []
    seqs = []

    # use sliding window to create sequences
    # TODO check the indices!
    for i in range(len(data_close) - M - N):
        X_seq.append(data_close[i : i + M])
        y_seq.append(data_close[i + M])

    # remove extra dimension placed in. 
    X_seq_np = np.array(X_seq).squeeze()
    y_seq_np = np.array(y_seq).squeeze()

    print("X_seq_np shape: ", X_seq_np.shape)

    # use sklearn to shuffle both and keep indices the same
    # this apparently does not pattern match. 
    shuffled = shuffle(X_seq_np, y_seq_np, random_state=rstate)
    print(len(shuffled))
    X_shuffle = shuffled[0]
    y_shuffle = shuffled[1]

    print("X_shuffle shape: ", X_shuffle.shape)


    # split 80/20
    split_index = int(np.size(X_shuffle, axis = 0) * 0.8)
    X_train = X_shuffle[:split_index, :]
    X_test = X_shuffle[split_index:, :]
    y_train = y_shuffle[:split_index]
    y_test = y_shuffle[split_index:]

    y_train

    print("X_train shape: ", np.shape(X_train))
    print("y_train shape: ", np.shape(y_train))
    print("X_test shape: ", np.shape(X_test))
    print("y_test shape: ", np.shape(y_test))

    # Scale the data
    # need to scale the labels too since we are using an activation function
    # need to add an extra dimension because sklearn is expecting a 2D matrix. 
    mms_X = MinMaxScaler()
    mms_y = MinMaxScaler()
    X_train = mms_X.fit_transform(X_train)
    y_train = mms_y.fit_transform(y_train.reshape(-1, 1)).squeeze()
    X_test = mms_X.transform(X_test)
    y_test  = mms_y.transform(y_test.reshape(-1, 1)).squeeze()

    # # convert to tensors 
    # X_train_tensor = torch.tensor(X_train, dtype = torch.float64)
    # y_train_tensor = torch.tensor(y_train, dtype = torch.float64)
    # X_test_tensor = torch.tensor(X_test, dtype = torch.float64)
    # y_test_tensor = torch.tensor(y_test, dtype = torch.float64)

    # # put inside tensor datasets
    # train_set = TensorDataset(X_train_tensor, y_train_tensor)
    # test_set = TensorDataset(X_test_tensor, y_test_tensor)

    # return [train_set, test_set]

    # a dict is much smarter here
    return {
        "X_train" : X_train, 
        "y_train" : y_train, 
        "X_test" : X_test, 
        "y_test" :  y_test
    }




data_nvda = yfinance.download('NVDA', start = '2021-01-01', end = '2021-12-31')
data_gme = yfinance.download('GME', start = '2021-01-01', end = '2021-12-31')
data_dji = yfinance.download('DJI', start = '2021-01-01', end = '2021-12-31')
data_ma = yfinance.download('MA', start = '2021-01-01', end = '2021-12-31')

stocks = [data_nvda, data_gme, data_dji, data_ma]
prepped_stocks = []

for stock in stocks:
    prepped_stocks.append(preprocess(stock))

# nvda_ten = torch.tensor(data_nvda, dtype=np.float64);
# gme_ten = torch.tensor(data_gme, dtype=np.float64);
# dji_ten = torch.tensor(data_dji, dtype=np.float64);
# ma_ten = torch.tensor(data_ma, dtype=np.float64);


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

X_seq_np shape:  (190, 60)
2
X_shuffle shape:  (190, 60)
X_train shape:  (152, 60)
y_train shape:  (152,)
X_test shape:  (38, 60)
y_test shape:  (38,)


ValueError: Expected 2D array, got 1D array instead:
array=[14.20719242 20.65119362 22.13021278 21.20118141 19.16409874 29.68923378
 14.57576466 22.16222954 20.76842499 19.75275612 21.6143074  15.22687054
 18.92290306 29.96165657 21.03054237 20.32518959 21.06846237 15.44906235
 14.10794163 17.41311264 17.97756577 19.90440178 27.74574852 25.51306534
 13.72391033 17.76954079 15.87249374 22.39365768 31.37252426 18.38513565
 15.32188225 22.33777809 30.26174164 15.23834038 19.54921532 20.24861717
 22.43058395 19.45442963 19.16160583 32.91542053 13.98051262 20.14186287
 22.03442001 19.01369286 19.85950851 16.22360611 15.43908978 14.28100491
 22.1052742  17.31857681 31.608181   31.43654251 20.62739563 33.30558777
 20.40785408 15.34307861 31.67903137 20.00442505 18.61460686 22.34875679
 22.19508553 15.66052818 14.27028275 16.92708778 32.3625412  22.61419487
 30.63200378 30.42940903 15.32312775 22.63614464 19.45842361 24.66485786
 29.01722145 22.01756096 22.29387283 15.13310909 22.43457222 32.6

2. Defining an RNN. Unlike the previous FNN models, we want to predict the prices in the future
N days based on the prices in the past M days (including today). Usually, M is much larger than N. Here,
you may simply set N to be 1 and choose an integer that is at least 50 for M. You are required to define a
neural network which has at least one recurrent layer for this purpose.

In [ ]:
# TODO implement the baseline model

class _RNN_(nn.Module):
    def __init__(self, input_size = 1, hidden_size = 64, output_size = 1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first = True)
        self.fully_connected_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, hidden_state = self.rnn(x)
        output = output[:, -1, :]
        output = self.fully_connected_layer(output)
        return output

def batch_data(X_train : np.ndarray, y_train : np.ndarray, n_batches = 10):
    X_train, y_train = shuffle(X_train, y_train, random_state=rstate)
    batch_size = int(X_train.shape[0] / n_batches)
    # this split code was taken from google
    X_batches = [X_train[i: i + batch_size] for i in range(0, X_train.shape[0], batch_size)]
    y_batches = [y_train[i: i + batch_size] for i in range(0, y_train.shape[0], batch_size)]

    return X_batches, y_batches

# NOTE the data is scaled, so we need to perform the inverse transform to map it back. 
def train_model(model, X_train, y_train, X_test, y_test, epochs=20):
    mean_squared_error = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

    losses = []

    training_start_time = time.time()

    X_batches, y_batches = batch_data(X_train, y_train, n_batches = 10)

    for epoch in range(epochs):
        epoch_loss = 0

        # extra inner loop to cover each minibatch
        # float64 is apparently not okay?
        # model expects a 3D tensor, (n_batches, n_samples, n_features), must unsqueeze.

        # training mode
        model.train()

        for X_batch, y_batch in zip(X_batches, y_batches):
            X_batch_t = torch.tensor(X_batch, dtype = torch.float32).unsqueeze(-1)
            y_batch_t = torch.tensor(y_batch, dtype = torch.float32).unsqueeze(-1)

            # clear gradients
            optimizer.zero_grad()
            predictions = model(X_batch_t)
            loss = mean_squared_error(predictions, y_batch_t)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        # must average the loss for batches.
        losses.append(epoch_loss / len(X_batches))
    training_end_time = time.time()

    training_time = training_end_time - training_start_time

    model.eval()
    X_test_t = torch.tensor(X_test, dtype = torch.float32).unsqueeze(-1)
    y_test_t = torch.tensor(y_test, dtype = torch.float32).unsqueeze(-1)
    with torch.no_grad():
        test_predictions = model(X_test_t)
        test_loss = mean_squared_error(test_predictions, y_test_t)
    
    return {
        "test_loss": test_loss.item(),
        "train_losses": losses,
        "training_time": training_time
    }

In [18]:
# create a new RNN for each 

print("GPU? ", torch.cuda.is_available())

print(type(prepped_stocks[0]))

nvda_RNN = _RNN_(1, 2, 1)
gme_RNN = _RNN_(1, 2, 1)
dji_RNN = _RNN_(1, 2, 1)
ma_RNN = _RNN_(1, 2, 1)

# for vals in prepped_stocks:
#     print("X_train shape: ", np.shape(vals["X_train"]))
#     print("y_train shape: ", np.shape(vals["y_train"]))
#     print("X_test shape: ",  np.shape(vals["X_test"]))
#     print("y_test shape: ",  np.shape(vals["y_test"]))

nets = [nvda_RNN, gme_RNN, dji_RNN, ma_RNN]

train_model(nvda_RNN, prepped_stocks[0]["X_train"], prepped_stocks[0]["y_train"], prepped_stocks[0]["X_test"], prepped_stocks[0]["y_test"])


GPU?  False
<class 'dict'>


{'test_loss': 452.7051696777344,
 'train_losses': [518.7742170854049,
  517.2628062855114,
  515.7515785910866,
  514.240253795277,
  512.7291370738636,
  511.2182090065696,
  509.7068620161577,
  508.1940945712003,
  506.67840576171875,
  505.15785078568894,
  503.6299133300781,
  502.0915832519531,
  500.5394259366122,
  498.9694935191761,
  497.377421985973,
  495.75839788263494,
  494.1073192249645,
  492.4190590598366,
  490.68861250443894,
  488.9118818803267],
 'training_time': 0.6994481086730957}